# Projet — différences entre la NAV et le prix de marché des ETF

## Problématique

Deux actifs peuvent représenter le même investissement — l'ETF et le portefeuille sous-jacent constitué par le fond, représenté par sa valeur nette des actifs (NAV) — tout en affichant des prix différents. L'objectif est de mesurer ces écarts, de voir si ils sont liés au régime de marché (volattilité et liquidité) et d'observer s'ils se corrigent rapidement.

L'étude se concentre sur 4 ETF : SPY, SPEM, SPIB et SPHY répliquant respectivement  le S&P500, les marchés actions émergents, un panier d'obligations Investment Grade (à faible risque de défaut), et un panier d'obligations High Yield (à forte probabilité de défaut).
On comparera la NAV quotidienne aux prix de ces 4 ETF.

Les étapes sont : (1) nettoyer les données de NAV et de prix de clôture, (2) calculer la prime/décote, (3) comparer les écarts selon la volatilité et la liquidité, puis (4) mesurer la correction le jour suivant.

## 1. Importation et nettoyage des données de NAV

La première cellule télécharge les historiques de NAV dans un dataframe, et les nettoie en retirant toutes les dates sans NAV. On préfère cela à de l'interpolation ou à une moyenne car on créerait alors artificiellement des écarts importants entre NAV et ETFs (les écarts étant usuellement très faibles).

La NAV (*Net Asset Value*) représente la valeur par part du portefeuille détenu par l'ETF et peut être résumée par la relation :

$$\mathrm{NAV}_t = \frac{\text{valeur des actifs du portefeuille à }t}{\text{nombre de parts de l'ETF à }t}$$



In [ ]:
#Récupération des données de NAVs (net asset value = valeur nette des actifs détenus par l'ETF,
#cad l'indice que réplique l'ETF)

import pandas as pd

url = "https://www.ssga.com/library-content/products/fund-data/etfs/us/navhist-us-en-{}.xlsx"

#S&P500
#Emerging Markets (S&P Emerging BMI Index), 
#Investement Grade Bonds (Bloomberg U.S. Intermediate Corporate Bond Index)
#High Yield Bonds  (ICE BofA US High Yield Index)
tickers = ["spy", "spem", "spib", "sphy"] 

nav = {}

#On importe les données (header = 3 car 3 premières lignes vides / usecols = A:D car 2 dernières colonnes vides à drop)
for ticker in tickers:
    nav[ticker] = pd.read_excel(url.format(ticker), header=3, usecols = "A:D").dropna(axis=1, how="all")


#On convertit toutes les dates en type datetime, et on abandonne toutes les lignes sans dates
#format indique le format des dates : ici 15-Oct-2026
#errors = coerce : transforme les textes qui ne sont pas des dates en NaT (valeur manquante pour les dates)
for ticker in tickers:
    nav[ticker]["Date"] = pd.to_datetime(nav[ticker]["Date"],format="%d-%b-%Y",errors="coerce")
    dates_manquantes = nav[ticker]["Date"].isna().sum()
    print(f"{ticker.upper()} - dates invalides ou manquantes :", dates_manquantes)
    nav[ticker] = nav[ticker].dropna(subset=["Date"]).reset_index(drop=True)
    nav[ticker]= nav[ticker].drop_duplicates(subset=["Date"]).sort_values("Date").reset_index(drop=True)

#On convertit toutes les autres informations numériques en type float
for ticker in tickers :
    nav[ticker]["NAV"] = pd.to_numeric(nav[ticker]["NAV"],errors="coerce")

#On fait le choix de simplement retirer toutes les dates sans données ou négatives pour la NAV
#(et non pas d'interpoler ou de remplacer les valeurs manquantes) 
#car les écarts entre NAVs et ETFs sont très faibles, et créer des données donnerait lieu à des écarts 
#artificiellement grands
#On compte également le nombre de valeurs supprimées sur ce critère, et le nbre de NAV manquantes après pour vérifier
for ticker in tickers :
    nbre_lignes_avant= len(nav[ticker])
    nav[ticker]=nav[ticker].dropna(subset=["NAV"])
    nav[ticker]=nav[ticker][nav[ticker]["NAV"]>0]
    nav[ticker]=nav[ticker].reset_index(drop=True)
    print("Nombre lignes supprimées car NAV manquante ou négative :", nbre_lignes_avant - len(nav[ticker]))
    print("Valeurs NAV manquantes après nettoyage :", nav[ticker]["NAV"].isna().sum())
    display(nav[ticker].head())

# Afficher la première et la dernière date pour chaque NAV : utile pour merge plus tard
for ticker in tickers:
    nav[ticker] = nav[ticker].sort_values("Date").reset_index(drop=True)
    print(ticker.upper())
    print("Première date :",nav[ticker]["Date"].iloc[0])
    print("Dernière date :",nav[ticker]["Date"].iloc[-1])
    print("Nombre d'observations :",len(nav[ticker]))
  


## 2. Importation des prix de marché et des volumes

On récupère les prix de clôture et les volumes des quatre ETF via Yahoo Finance. Le prix de marché est celui auquel l'ETF s'échange en bourse. Les volumes nous serveront de proxy pour calculer la liquidité d'un ETF. 
Comme précédemment, on nettoie le dataframe et on en retire les valeurs manquantes.

Le prix utilisé est le cours de clôture :

$$P_t^{\mathrm{marché}} = \mathrm{Close}_t$$


In [ ]:
#Récupération des prix (de clôture), aussi appelés "Close" ici dans le code, des ETFs

import yfinance as yf
import pandas as pd

etfs = ["SPY", "SPEM", "SPIB", "SPHY"]

prix = {}

#autoadjust = False   # prix de marché historique, qui ne prend pas rétroactivement en compte les distributions de dividendes
#EX : cad que si actif(t1) = 100, dividende = 2 => actif(t2) = 98   : l'actif(t1) = 100 et pas 98 avec False
#progress = False #uniquement une question de présentation
#la dernière ligne permet de résoudre un pb de yahho finance : EX : prix[ticker]["Close"] est un dataframe et non une series
for ticker in etfs:
    prix[ticker] = yf.download(ticker, start="2012-06-19", end="2026-10-15", auto_adjust=False, 
    progress=False)
    if isinstance(prix[ticker].columns, pd.MultiIndex):
        prix[ticker].columns = (prix[ticker].columns.get_level_values(0))

#nettoyage de la colonne de dates : comptage des dates manquantes, suprression des lignes aux dates manquantes
for ticker in etfs :
    prix[ticker].index = pd.to_datetime(prix[ticker].index, errors ="coerce")
    dates_manquantes = prix[ticker].index.isna().sum()
    print(f"{ticker} - dates invalides ou manquantes :", dates_manquantes)
    prix[ticker] = prix[ticker][prix[ticker].index.notna()]  #car la colonne avec les dates est l'index et non pas une colonne régulière
    prix[ticker].index.name = "Date"

#suppression de toutes les lignes ne contenant pas de prix close, ou un prix close négatif
for ticker in etfs :
    nombre_lignes_avant = len(prix[ticker])
    prix[ticker]["Close"] = pd.to_numeric(prix[ticker]["Close"], errors="coerce")
    prix[ticker]=prix[ticker].dropna(subset=["Close"])
    prix[ticker]=prix[ticker][prix[ticker]["Close"]>=0]
    prix[ticker] = prix[ticker].sort_index().reset_index()
    print("Nombre lignes supprimées car Close manquant ou négatif :", nbre_lignes_avant - len(prix[ticker]))
    print("Valeurs NAV manquantes après nettoyage :", prix[ticker]["Close"].isna().sum())
    display(prix[ticker][["Close", "Volume"]].head())

# Afficher la première et la dernière date pour chaque ETF : utile pour merge plus tard
for ticker in tickers:
    nav[ticker] = nav[ticker].sort_values("Date").reset_index(drop=True)
    print(ticker.upper())
    print("Première date :",nav[ticker]["Date"].iloc[0])
    print("Dernière date :",nav[ticker]["Date"].iloc[-1])
    print("Nombre d'observations :",len(nav[ticker]))

for ticker in etfs:
    prix[ticker] = prix[ticker].sort_values("Date").reset_index(drop=True)
    print(ticker)
    print("Première date :", prix[ticker]["Date"].iloc[0])
    print("Dernière date :", prix[ticker]["Date"].iloc[-1])
    print("Nombre d'observations :", len(prix[ticker]))
    print()

## 3. Fusion des dataframes NAV et ETF et calcul des primes et décotes.

La fusion ne conserve que les dates communes entre NAV et prix de clôture. Pour chaque date, l'écart relatif est calculé ainsi :

$$\mathrm{Prime/Décote}_t = \frac{P_t^{\mathrm{marché}} - \mathrm{NAV}_t}{\mathrm{NAV}_t} \times 100$$

Une valeur positive correspond à une prime et une valeur négative à une décote. On calcule aussi l'écart absolu :

$$\mathrm{Écart\ absolu}_t = \left|\mathrm{Prime/Décote}_t\right|$$

L'ajustement du split 2-pour-1 de SPEM avant le 16 octobre 2017 évite de confondre une rupture technique avec un véritable écart économique.

> Convention : une prime/décote positive signifie que le prix de marché est supérieur à la NAV ; une valeur négative correspond à une décote.

In [ ]:
#Fusion des NAV et prix de marché : on conserve uniquement les dates communes à la NAV et au prix de clôture.
#Le tableau contient donc une ligne par ETF et par date, avec le prix, la NAV, le volume et la prime / décôte
#on conserve les volumes pour avoir une mesure de la liqudité, pour plus tard

type_etf = {"SPY": "Actions américaines", "SPEM": "Actions émergentes", "SPIB": "Obligations investment grade", "SPHY": "Obligations high yield"}
classe_etf = {"SPY": "Actions", "SPEM": "Actions", "SPIB": "Obligations", "SPHY": "Obligations"}

tableaux = []

for ticker in etfs : 
    nav_merge = nav[ticker.lower()][["Date", "NAV"]].copy()
    # 2 lignes suivantes : Ajustement du split 2 pour 1 de SPEM effectif le 16 octobre 2017 : expliquer pourquoi :
    #On avait 50% de différence de valeur pour SPEM entre la NAV de et l'ETF de yahoo finance et la NAV de State Street : le 2e ne prenait pas en compte le stock split
    if ticker == "SPEM":
        nav_merge.loc[nav_merge["Date"] < pd.Timestamp("2017-10-16"), "NAV"] /= 2
    prix_merge = prix[ticker].reindex(columns=["Date", "Close", "Volume"]).copy()

    jointure = pd.merge(nav_merge,prix_merge, on = "Date", how = "inner")
    print(len(jointure))
    jointure["ETF"] = ticker
    jointure["Type"] = type_etf[ticker]
    jointure["Classe"] = classe_etf[ticker]
    jointure["Prime_decote_pct"] = (jointure["Close"] - jointure["NAV"]) / jointure["NAV"] * 100
    jointure["Abs_prime_decote_pct"] = jointure["Prime_decote_pct"].abs()
    jointure["Prime_decote_moyenne_30j"]= jointure["Prime_decote_pct"].rolling(30, min_periods=5).mean()
    tableaux.append(jointure)

donnees = pd.concat(tableaux, ignore_index = True)
donnees = donnees.sort_values(["ETF", "Date"]).reset_index(drop=True)

Statistiques_ecarts = (donnees.groupby(["ETF", "Classe", "Type"]).agg(
        Nombre_observations=("Date", "size"),
        Prime_decote_moyenne_pct=("Prime_decote_pct", "mean"),
        Prime_decote_mediane_pct=("Prime_decote_pct", "median"),
        Ecart_absolu_moyen_pct=("Abs_prime_decote_pct", "mean"),
        Ecart_absolu_max_pct=("Abs_prime_decote_pct", "max")
    ).reset_index())

#on veut regarder quand est-ce que les écarts NAV-ETF dépassent certains seuils (0.25%, 0.5% et 1%)
for seuil in [0.25, 0.5, 1.0]:
        #calcule la proportion de jours en % pour lesquels on dépasse chaque seuil pour chaque ETF 
        depassement = donnees.groupby("ETF")["Abs_prime_decote_pct"].apply(lambda s: (s > seuil).mean() * 100)
        #On rajoute une colonne dans laquelle on trouve le nbre de jours au dessus de chaque seuil pour chaque ETF dans le dataframe de stats
        Statistiques_ecarts[f"Jours_au_dessus_{str(seuil)}pct"] = Statistiques_ecarts["ETF"].map(depassement)

display(donnees[["ETF", "Date", "Close", "NAV", "Volume", "Prime_decote_pct", "Prime_decote_moyenne_30j"]].head())
display(Statistiques_ecarts.round(4)) 


#On affiche les primes et décôtes quotidiennes
import matplotlib.pyplot as plt

plt.style.use("ggplot")

fig, axes = plt.subplots(2, 2, figsize=(15, 9)) #on veut 4 graphes, 2 en ligne, 2 en colonne
axes = axes.flatten()


for ax, (ticker, groupe) in zip(axes, donnees.groupby("ETF", sort=False)): #cette boucle associe un emplacement de graphique "ax", un ETF "ticker", et les données correspondantes "groupe"
    groupe = groupe.sort_values("Date")

    ax.plot(groupe["Date"], groupe["Prime_decote_pct"], linewidth=0.8)  #trace

    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"Prime/décote de {ticker}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Écart (%)")
    ax.grid(True)

fig.suptitle("Évolution de la prime/décote par ETF", fontsize=16) #titre des 4 graphes ensemble
plt.tight_layout()
plt.show()

### Résultats de la fusion des données

La fusion produit **3 601 dates communes par ETF**. L'écart absolu moyen est de **0,0242 % pour SPY**, **0,1315 % pour SPIB**, **0,3336 % pour SPHY** et **0,4514 % pour SPEM**.

La proportion de jours où l'écart dépasse 0,25 % est respectivement de **0,31 %**, **15,33 %**, **45,40 %** et **63,68 %** pour SPY, SPIB, SPHY et SPEM.

SPY et SPIB suivent très étroitement sa NAV, alors que SPEM et SPHY présentent les écarts les plus fréquents entre NAV et valeur de l'ETF.

On peut attribuer cette différence au fait que les sous-jacents de SPEM et SPHY sont généralement moins liquides, ce qui rend leur achat et vente par le détenteur du fond pour recalibrer sa réplication de l'ETF plus difficile en cas de stress de marché.

Les pics observés en 2020, notamment en mars, correspondent au choc de la pandémie de Covid-19 : la forte volatilité, les ventes massives et la baisse de la liquidité ont temporairement amplifié les primes et décotes, surtout pour SPEM et SPHY.

## 4. Volatilité, liquidité et ampleur des écarts

On se demande ici si les primes et décotes importants, c'est-à-dire les écarts importants entre NAV et prix de l'ETF sont liés aux conditions de marché (volatilité et liquidité).

Le VIX sert à distinguer les régimes de volatilité faible et forte. La règle appliquée est :

$$R_{\mathrm{vol},t} = \begin{cases} \text{Faible} & \text{si } VIX_t < 15 \\ \text{Forte} & \text{si } VIX_t \geq 30 \\ \text{Intermédiaire} & \text{sinon} \end{cases}$$

Pour la liquidité, les quartiles sont calculés séparément pour chaque ETF :

$$R_{\mathrm{liq},t} = \begin{cases} \text{Faible} & \text{si } Volume_t \leq Q_{25} \\ \text{Forte} & \text{si } Volume_t \geq Q_{75} \\ \text{Intermédiaire} & \text{sinon} \end{cases}$$

La cellule compare ensuite l'écart absolu moyen entre les régimes faible et fort, sans prétendre établir une causalité.

In [ ]:
#Evolution des écarts de comparaison volatilité / liquidité : on tente de voir si les écarts entre NAV et ETF sont liés à des conditions de marché
#La volatilité sera mesurée par le VIX, indice de la vol implicite du S&P500 sur 30j

import numpy as np

vix = yf.download("^VIX", start=donnees["Date"].min(), end=donnees["Date"].max(), auto_adjust=False, progress=False)
#Le VIX de yahoo finance a un multiIndex, cad que le nom de chaque colonne est un couple ( , ) : sol° : 
if isinstance(vix.columns, pd.MultiIndex):
    vix.columns = vix.columns.get_level_values(0)
#On clean VIX : on renomme la colonne, on passe en format datetime, et on retire les valeurs manquantes et doubles
vix = vix.reset_index().rename(columns={"Close": "VIX"})
vix["Date"] = pd.to_datetime(vix["Date"], errors="coerce")
vix = vix[["Date", "VIX"]].dropna().drop_duplicates("Date")

donnees = donnees.merge(vix, on="Date", how="left")

#On étiquette les régimes :  haute et faible volatilité (on fait l'hypothèse que haute <=> >30 et basse <=> <15)
#Pour ce qui est de la liquidité, il faut étiquetter chaque ETF séparément, pour ne pas comparer des 
#volumes absolus de fonds différents

#"select" attribue une catégorie selon plusieurs conditions : ici si VIX<15, faible, entre 15 et 30, intermédiaire, au dessus, forte
donnees["Regime_volatilite"] = np.select([donnees["VIX"] < 15, donnees["VIX"] >= 30],
    ["Faible", "Forte"], default="Intermédiaire")

groupes_regimes = []
#groupe donne le dataframe associé à l'ETF "ticker", selon la logique suivante : 
# ticker = "SPY" => groupe = donnees[donnees["ETF"] == "SPY"])
for ticker, groupe in donnees.groupby("ETF", sort=False):
    groupe = groupe.copy()
    #maintenant, les régimes de liquidité, en utilisant les volumes des ETFs 
    vol_q25, vol_q75 = groupe["Volume"].quantile([0.25, 0.75])
    groupe["Regime_liquidite"] = np.select([groupe["Volume"] <= vol_q25, groupe["Volume"] >= vol_q75],
        ["Faible", "Forte"], default="Intermédiaire")
    groupes_regimes.append(groupe)
donnees = pd.concat(groupes_regimes, ignore_index=True).sort_values(["ETF", "Date"])


#On prend toutes les données telles que la vol est faible ou forte, puis on les regroupe par les 4 types
#(oblig ou actions, risquées ou stable), puis on calcule quelques statistiques pour chacun
comparaison_volatilite = (donnees[donnees["Regime_volatilite"].isin(["Faible", "Forte"])]
    .groupby(["Type", "Regime_volatilite"])
    .agg(Nb_jours=("Date", "size"), Prime_moyenne_pct=("Prime_decote_pct", "mean"),
        Ecart_absolu_moyen_pct=("Abs_prime_decote_pct", "mean"))
    .reset_index())

display(comparaison_volatilite.round(4))

#On fait exactement pareil, mais pour la liquidité : 
comparaison_liquidite = (donnees[donnees["Regime_liquidite"].isin(["Faible", "Forte"])]
    .groupby(["Type", "Regime_liquidite"])
    .agg(Nb_jours=("Date", "size"), Prime_moyenne_pct=("Prime_decote_pct", "mean"),
        Ecart_absolu_moyen_pct=("Abs_prime_decote_pct", "mean"))
    .reset_index())

display(comparaison_liquidite.round(4))


import matplotlib.pyplot as plt

# Écart moyen selon le régime de volatilité
graph_volatilite = (donnees[donnees["Regime_volatilite"].isin(["Faible", "Forte"])]
    .groupby(["ETF", "Regime_volatilite"])["Abs_prime_decote_pct"]
    .mean().unstack().reindex(columns=["Faible", "Forte"]))

# Écart moyen selon le régime de liquidité
graph_liquidite = (donnees[donnees["Regime_liquidite"].isin(["Faible", "Forte"])]
    .groupby(["ETF", "Regime_liquidite"])["Abs_prime_decote_pct"]
    .mean().unstack().reindex(columns=["Faible", "Forte"]))

#Modélisation : comparaison entre volatilité faible et forte, et liquidité faible et forte
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

graph_volatilite.plot(kind="bar", ax=axes[0], color=["blue", "red"])

axes[0].set_title("Écart absolu moyen selon la volatilité")
axes[0].set_xlabel("ETF")
axes[0].set_ylabel("Écart absolu moyen (%)")
axes[0].legend(title="Volatilité")
axes[0].grid(axis="y")

graph_liquidite.plot(kind="bar", ax=axes[1], color=["blue", "red"])

axes[1].set_title("Écart absolu moyen selon la liquidité")
axes[1].set_xlabel("ETF")
axes[1].set_ylabel("Écart absolu moyen (%)")
axes[1].legend(title="Liquidité")
axes[1].grid(axis="y")

plt.tight_layout()
plt.show()

### Résultats de l'analyse des conditions de marché

En forte volatilité, l'écart absolu moyen augmente pour les quatre catégories : **0,0202 % → 0,0671 %** pour les actions américaines, **0,3752 % → 0,8283 %** pour les actions émergentes, **0,1314 % → 0,3412 %** pour l'investment grade et **0,3323 % → 0,5096 %** pour le high yield. L'effet est particulièrement marqué pour SPEM.

L'effet du volume est hétérogène : l'écart augmente lorsque le volume est fort pour les actions américaines et émergentes, mais diminue pour les obligations (**0,4625 % → 0,1722 %** pour le high yield ; **0,2503 % → 0,0618 %** pour l'investment grade). Le volume est donc un proxy imparfait de la liquidité.

## 5. Correction le jour suivant

On se demande si les écarts se rétablissent le jour d'après, quand les opérateurs de marché arbitrent la différence entre la valeur de la NAV et des ETF.

On retient les écarts absolus d'au moins 0,25 %, puis on compare leur valeur avec celle de la séance suivante. La variation de l'écart est définie par :

$$\Delta_t = \mathrm{Écart\ absolu}_{t+1} - \mathrm{Écart\ absolu}_t$$

Une correction est comptée lorsque :

$$\mathrm{Correction}_t = \mathbb{1}(\Delta_t < 0)$$

Le taux de correction est ensuite :

$$\mathrm{Taux\ de\ correction} = \frac{\text{nombre de corrections}}{\text{nombre d'écarts importants observés}} \times 100$$

Dans le nuage de points, les observations sous la diagonale correspondent à une correction et celles au-dessus à une aggravation de l'écart enregistré le jour précédent.

In [ ]:
#Les écarts se corrigent-ils le jour suivant ? 

# Seuil à partir duquel on considère l'écart comme important (on a calculé au dessus ça représente <2% de l'échantillon des jours)
seuil_ecart = 0.25

# Trier les données par ETF et par date
donnees = donnees.sort_values(["ETF", "Date"]).copy()

# Écart absolu observé le jour suivant, sans mélanger les ETF
ecart_lendemain = donnees.groupby("ETF")["Abs_prime_decote_pct"].shift(-1)

# Variation de l'écart entre aujourd'hui et demain
donnees["Variation_abs_1j"] = ecart_lendemain - donnees["Abs_prime_decote_pct"]

# True si l'écart diminue le lendemain, la dernière date de chaque ETF n'a pas de lendemain
donnees["Correction_lendemain"] = np.where(ecart_lendemain.notna(), donnees["Variation_abs_1j"] < 0, np.nan)

# On garde uniquement les écarts importants
donnees_ecarts_importants = donnees[(donnees["Abs_prime_decote_pct"] >= seuil_ecart) & (ecart_lendemain.notna())].copy()

# Résumé des corrections
vitesse_correction = (donnees_ecarts_importants.groupby(["ETF", "Classe", "Type"])
    .agg(Nb_dates=("Correction_lendemain", "count"),
        Taux_correction_pct=("Correction_lendemain", lambda x: x.mean() * 100),
        Variation_abs_moyenne_1j=("Variation_abs_1j","mean"))
    .reset_index())

display(vitesse_correction.round(4))


#Modélisation 

# Calcul de l'écart du lendemain
donnees_ecarts_importants["Ecart_abs_1j"] = (donnees_ecarts_importants["Abs_prime_decote_pct"]
    + donnees_ecarts_importants["Variation_abs_1j"])

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
axes = axes.flatten()

for ax, (ticker, groupe) in zip(axes, donnees_ecarts_importants.groupby("ETF")):

    ax.scatter(groupe["Abs_prime_decote_pct"], groupe["Ecart_abs_1j"], alpha=0.35, s=12)

    valeur_max = max(groupe["Abs_prime_decote_pct"].max(), groupe["Ecart_abs_1j"].max())

    # Ligne indiquant une absence de changement
    ax.plot([0, valeur_max], [0, valeur_max], color="black", linestyle="--")

    ax.set_title(f"Correction des écarts : {ticker}")
    ax.set_xlabel("Écart absolu aujourd'hui (%)")
    ax.set_ylabel("Écart absolu le lendemain (%)")
    ax.grid(True)

fig.suptitle("Évolution des écarts importants entre deux séances", fontsize=15)

plt.tight_layout()
plt.show()

#Au dessus de la diagonale : correction, en dessous, aggravation de l'écart

### Résultats de l'étude des corrections

Le taux de correction des écarts importants est de **68,67 % pour SPEM** (variation moyenne : −0,1663 point de pourcentage), **67,28 % pour SPHY** (−0,1050), **65,22 % pour SPIB** (−0,0526) et **100 % pour SPY** (−0,3203).

On observe donc que dans la majorité des cas les arbitrages mènent les écarts à se résorber.

Le résultat de SPY doit être interprété avec prudence car il ne repose que sur **11 dates**. Dans l'ensemble, les écarts importants ont tendance à diminuer dès la séance suivante, sans disparaître systématiquement.

# Conclusion et limites

## Conclusion

L'ETF et le portefeuille sous-jacent peuvent représenter la même exposition tout en ayant des prix différents, car le prix de l'ETF est déterminé en continu par l'offre et la demande alors que la NAV est calculée à partir de la valorisation des actifs du portefeuille, souvent à un autre moment et avec d'autres conventions. Les résultats montrent que les écarts sont faibles pour SPY, plus importants pour SPEM et SPHY, qu'ils s'amplifient généralement lorsque la volatilité est forte et qu'ils se corrigent souvent dès le lendemain.

## Limites du projet

- La NAV et le prix de clôture peuvent correspondre à des moments différents de la même journée.
- Le VIX est un indicateur de volatilité du S&P 500, pas une mesure spécifique de chacun des ETF.
- Les seuils `0,25 %`, `0,5 %`, `1 %`, `VIX < 15` et `VIX ≥ 30` sont des choix d'analyse ; ils ne constituent pas des seuils universels.
- Le volume est un proxy imparfait de la liquidité.
- Le projet ne rend pas compte de causalité mais de tendances.